# Parallel Asteroid Belt Dynamics & Planet Formation Simulation
## Serial C++ Implementation

**A research-grade N-body gravitational simulation of protoplanetary disk evolution**

---
## Physics Model

### Newtonian Gravity

Every body $i$ experiences a gravitational force from every other body $j$:

$$\vec{F}_{ij} = \frac{G \, m_i \, m_j}{|\vec{r}_{ij}|^2 + \epsilon^2} \hat{r}_{ij}$$

where:
- $G$ is the gravitational constant
- $\vec{r}_{ij} = \vec{r}_j - \vec{r}_i$ is the displacement vector
- $\epsilon$ is a **softening length** that prevents singularities at close approach

The total acceleration on body $i$:

$$\vec{a}_i = \sum_{j \neq i} \frac{G \, m_j}{(|\vec{r}_{ij}|^2 + \epsilon^2)^{3/2}} \vec{r}_{ij}$$

### Integration Methods

**Euler Method** (1st order) — simple but drifts energy:
$$\vec{v}(t + \Delta t) = \vec{v}(t) + \vec{a}(t) \Delta t$$
$$\vec{r}(t + \Delta t) = \vec{r}(t) + \vec{v}(t) \Delta t$$

**Velocity-Verlet / Leapfrog** (2nd order, symplectic) — conserves energy over long timescales:

1. Half-kick: $\vec{v}(t + \tfrac{\Delta t}{2}) = \vec{v}(t) + \tfrac{\Delta t}{2} \vec{a}(t)$
2. Drift: $\vec{r}(t + \Delta t) = \vec{r}(t) + \Delta t \, \vec{v}(t + \tfrac{\Delta t}{2})$
3. Compute new forces → $\vec{a}(t + \Delta t)$
4. Half-kick: $\vec{v}(t + \Delta t) = \vec{v}(t + \tfrac{\Delta t}{2}) + \tfrac{\Delta t}{2} \vec{a}(t + \Delta t)$

**We use Velocity-Verlet** because it is:
- Time-reversible (symplectic)
- 2nd-order accurate
- Preserves orbital energy over thousands of timesteps
- Standard in molecular dynamics and astrophysical N-body codes

---
## Data Structures

### Body Representation

```cpp
struct Body {
    double x, y;       // position
    double vx, vy;     // velocity
    double ax, ay;     // acceleration
    double mass;
    double radius;
    bool   alive;      // false = merged into another body
};
```

### Memory Layout: AoS vs SoA

**Array of Structs (AoS):** `Body bodies[N]` — each body's data is contiguous in memory. Good for per-body operations and readability.

**Struct of Arrays (SoA):**
```cpp
double x[N], y[N], vx[N], vy[N], ax[N], ay[N], mass[N], radius[N];
```
Better for SIMD vectorization: the force loop touches `x[j], y[j], mass[j]` for all j — having these packed contiguously enables hardware vector units.

**Our choice:** AoS for the serial version (clarity and correctness first). The OpenMP version will also use AoS. A future CUDA port would benefit from SoA for coalesced memory access.

### Softening Parameter

We add $\epsilon^2$ to the denominator to:
1. Prevent division by zero when bodies are very close
2. Model the finite size of bodies (they are not point masses)
3. Stabilize the integrator — without softening, close encounters produce enormous accelerations that blow up the simulation

---
## Serial N-Body Simulation

The complete serial implementation below includes:
- **Step 4:** Full serial N-body algorithm with Velocity-Verlet integration — $O(N^2)$ force computation
- **Step 5:** Collision detection and momentum-conserving merging
- **Step 6:** Keplerian disk initial conditions with random perturbations
- **Step 7:** CSV output at configurable snapshot intervals

### Computational Complexity

The force computation is the bottleneck: every body interacts with every other → $O(N^2)$ per timestep. For $N = 2000$ bodies and $T = 1000$ steps, that is $2000^2 \times 1000 = 4 \times 10^9$ force evaluations. This is exactly why parallelization matters.

### Collision Model

When two bodies overlap ($d < r_1 + r_2$):
- **Mass conservation:** $m_{\text{new}} = m_1 + m_2$
- **Momentum conservation:** $\vec{v}_{\text{new}} = \frac{m_1 \vec{v}_1 + m_2 \vec{v}_2}{m_1 + m_2}$
- **Radius:** $r_{\text{new}} = \left(\frac{3(m_1 + m_2)}{4\pi\rho}\right)^{1/3}$ (assuming uniform density)

### Initial Conditions

Bodies are distributed in a disk with Keplerian orbital velocities:

$$v_{\text{circular}} = \sqrt{\frac{G \, M_\star}{r}}$$

with small random perturbations (5–10%) to seed instabilities. Without perturbations, the disk would remain in a perfectly stable orbit forever — the perturbations mimic real turbulence and allow gravitational instabilities to develop.

In [ ]:
%%writefile nbody_serial.cpp
// =============================================================================
// Parallel Asteroid Belt Dynamics & Planet Formation Simulation
// Serial C++ Implementation — Velocity-Verlet N-body with Collisions
// =============================================================================



// ========================== PHYSICS: FORCE COMPUTATION =======================
// Compute gravitational accelerations — O(N^2) all-pairs
void compute_forces(std::vector<Body>& bodies, double softening_sq) {
    int n = bodies.size();

    // Reset accelerations
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        bodies[i].ax = 0.0;
        bodies[i].ay = 0.0;
    }

    // All-pairs force computation
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        for (int j = i + 1; j < n; ++j) {
            if (!bodies[j].alive) continue;

            double dx = bodies[j].x - bodies[i].x;
            double dy = bodies[j].y - bodies[i].y;
            double dist_sq = dx * dx + dy * dy + softening_sq;
            double inv_dist = 1.0 / std::sqrt(dist_sq);
            double inv_dist3 = inv_dist * inv_dist * inv_dist;

            // Force magnitude / distance = G * m_j / r^3 (for acceleration on i)
            double fx = G * dx * inv_dist3;
            double fy = G * dy * inv_dist3;

            // Newton's 3rd law: apply equal-opposite
            bodies[i].ax += fx * bodies[j].mass;
            bodies[i].ay += fy * bodies[j].mass;
            bodies[j].ax -= fx * bodies[i].mass;
            bodies[j].ay -= fy * bodies[i].mass;
        }
    }
}

// ========================== INTEGRATOR: VELOCITY-VERLET ======================
void integrate_step(std::vector<Body>& bodies, double dt, double softening_sq) {
    int n = bodies.size();

    // Step 1: Half-kick (update velocities by half-step using current accelerations)
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        bodies[i].vx += 0.5 * dt * bodies[i].ax;
        bodies[i].vy += 0.5 * dt * bodies[i].ay;
    }

    // Step 2: Drift (update positions using half-kicked velocities)
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        bodies[i].x += dt * bodies[i].vx;
        bodies[i].y += dt * bodies[i].vy;
    }

    // Step 3: Compute new forces at the new positions
    compute_forces(bodies, softening_sq);

    // Step 4: Half-kick again (complete velocity update)
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        bodies[i].vx += 0.5 * dt * bodies[i].ax;
        bodies[i].vy += 0.5 * dt * bodies[i].ay;
    }
}

// ========================== COLLISION DETECTION & MERGING ====================
int handle_collisions(std::vector<Body>& bodies) {
    int n = bodies.size();
    int merges = 0;

    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        if (i == 0) continue;  // star does not participate in merging
        for (int j = i + 1; j < n; ++j) {
            if (!bodies[j].alive) continue;

            double dx = bodies[j].x - bodies[i].x;
            double dy = bodies[j].y - bodies[i].y;
            double dist = std::sqrt(dx * dx + dy * dy);
            double overlap_dist = bodies[i].radius + bodies[j].radius;

            if (dist < overlap_dist) {
                // Merge j into i (keep the more massive body's index)
                double m1 = bodies[i].mass;
                double m2 = bodies[j].mass;
                double m_total = m1 + m2;

                // Momentum-conserving velocity
                bodies[i].vx = (m1 * bodies[i].vx + m2 * bodies[j].vx) / m_total;
                bodies[i].vy = (m1 * bodies[i].vy + m2 * bodies[j].vy) / m_total;

                // Center of mass position (weighted)
                bodies[i].x = (m1 * bodies[i].x + m2 * bodies[j].x) / m_total;
                bodies[i].y = (m1 * bodies[i].y + m2 * bodies[j].y) / m_total;

                bodies[i].mass   = m_total;
                bodies[i].radius = compute_radius(m_total) * RADIUS_INFLATION;

                // Kill body j
                bodies[j].alive = false;
                ++merges;
            }
        }
    }
    return merges;
}

// ========================== I/O: CSV OUTPUT ==================================
void write_snapshot(const std::vector<Body>& bodies, int step, const std::string& dir) {
    std::ostringstream filename;
    filename << dir << "/step_" << std::setw(6) << std::setfill('0') << step << ".csv";

    std::ofstream file(filename.str());
    if (!file.is_open()) {
        std::cerr << "ERROR: Cannot open " << filename.str() << std::endl;
        return;
    }

    file << "id,x,y,vx,vy,mass,radius\n";
    int id = 0;
    for (size_t i = 0; i < bodies.size(); ++i) {
        if (!bodies[i].alive) continue;
        file << id << ","
             << std::scientific << std::setprecision(8)
             << bodies[i].x << ","
             << bodies[i].y << ","
             << bodies[i].vx << ","
             << bodies[i].vy << ","
             << bodies[i].mass << ","
             << bodies[i].radius << "\n";
        ++id;
    }
    file.close();
}

// Count alive bodies
int count_alive(const std::vector<Body>& bodies) {
    int count = 0;
    for (size_t i = 0; i < bodies.size(); ++i) {
        if (bodies[i].alive) ++count;
    }
    return count;
}

// ========================== MAIN =============================================
int main(int argc, char* argv[]) {
    // ----- Default simulation parameters -----
    SimParams params;
    params.num_bodies             = 2000;
    params.star_mass              = 1.989e30;        // 1 solar mass
    params.disk_inner             = 1.0e11;           // ~0.67 AU
    params.disk_outer             = 5.0e11;           // ~3.3 AU
    params.dt                     = 1.0e6;            // ~11.6 days
    params.num_steps              = 200000;
    params.snapshot_interval      = 10;
    params.softening              = 1.0e9;            // softening length
    params.velocity_perturbation  = 0.10;
    params.output_dir             = "data_serial";

    // Simple command-line overrides
    for (int i = 1; i < argc; ++i) {
        std::string arg = argv[i];
        if (arg == "--bodies"    && i+1 < argc) params.num_bodies        = std::atoi(argv[++i]);
        else if (arg == "--steps"     && i+1 < argc) params.num_steps         = std::atoi(argv[++i]);
        else if (arg == "--dt"        && i+1 < argc) params.dt               = std::atof(argv[++i]);
        else if (arg == "--interval"  && i+1 < argc) params.snapshot_interval = std::atoi(argv[++i]);
        else if (arg == "--outdir"    && i+1 < argc) params.output_dir       = argv[++i];
    }

    double softening_sq = params.softening * params.softening;

    std::cout << "=== N-Body Asteroid Belt Simulation (Serial) ===" << std::endl;
    std::cout << "Bodies: " << params.num_bodies << std::endl;
    std::cout << "Steps:  " << params.num_steps  << std::endl;
    std::cout << "dt:     " << params.dt << " s" << std::endl;
    std::cout << "Output: " << params.output_dir << "/" << std::endl;

    // Create output directory
    std::string cmd = "mkdir -p " + params.output_dir;
    system(cmd.c_str());

    // Generate initial conditions
    auto bodies = generate_initial_conditions(params);
    std::cout << "Generated " << bodies.size() << " bodies (1 star + "
              << params.num_bodies << " asteroids)" << std::endl;

    // Initial force computation (needed for Verlet first half-kick)
    compute_forces(bodies, softening_sq);

    // Write initial snapshot
    write_snapshot(bodies, 0, params.output_dir);

    // ----- Main simulation loop -----
    auto t_start = std::chrono::high_resolution_clock::now();

    for (int step = 1; step <= params.num_steps; ++step) {
        // Velocity-Verlet integration step
        integrate_step(bodies, params.dt, softening_sq);

        // Collision detection and merging
        int merges = handle_collisions(bodies);

        // Snapshot output
        if (step % params.snapshot_interval == 0) {
            write_snapshot(bodies, step, params.output_dir);
        }

        // Progress reporting
        if (step % 100 == 0 || step == params.num_steps) {
            int alive = count_alive(bodies);
            std::cout << "Step " << step << "/" << params.num_steps
                      << " | Bodies alive: " << alive;
            if (merges > 0) std::cout << " | Merges this step: " << merges;
            std::cout << std::endl;
        }
    }

    auto t_end = std::chrono::high_resolution_clock::now();
    double elapsed = std::chrono::duration<double>(t_end - t_start).count();

    std::cout << "\n=== Simulation Complete ===" << std::endl;
    std::cout << "Total time: " << std::fixed << std::setprecision(3) << elapsed << " s" << std::endl;
    std::cout << "Final alive bodies: " << count_alive(bodies) << std::endl;
    std::cout << "Snapshots saved to: " << params.output_dir << "/" << std::endl;

    return 0;
}

Writing nbody_serial.cpp


In [ ]:
!g++ -O3 -std=c++17 -o nbody_serial nbody_serial.cpp -lm
print("Compilation successful!")

nbody_serial.cpp: In function ‘int main(int, char**)’:
nbody_serial.cpp:302:11: warning: ignoring return value of ‘int system(const char*)’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
  302 |     system(cmd.c_str());
      |     ~~~~~~^~~~~~~~~~~~~
Compilation successful!


In [ ]:
!./nbody_serial --bodies 2000 --steps 200000 --interval 10 --outdir data_serial

=== N-Body Asteroid Belt Simulation (Serial) ===
Bodies: 2000
Steps:  200000
dt:     1e+06 s
Output: data_serial/
Generated 2001 bodies (1 star + 2000 asteroids)
Step 100/200000 | Bodies alive: 1968
Step 200/200000 | Bodies alive: 1933
Step 300/200000 | Bodies alive: 1897
Step 400/200000 | Bodies alive: 1863
Step 500/200000 | Bodies alive: 1837
Step 600/200000 | Bodies alive: 1806
Step 700/200000 | Bodies alive: 1779
Step 800/200000 | Bodies alive: 1749
Step 900/200000 | Bodies alive: 1723 | Merges this step: 1
Step 1000/200000 | Bodies alive: 1702
Step 1100/200000 | Bodies alive: 1682
Step 1200/200000 | Bodies alive: 1664
Step 1300/200000 | Bodies alive: 1641
Step 1400/200000 | Bodies alive: 1614
Step 1500/200000 | Bodies alive: 1589
Step 1600/200000 | Bodies alive: 1570
Step 1700/200000 | Bodies alive: 1551
Step 1800/200000 | Bodies alive: 1529 | Merges this step: 1
Step 1900/200000 | Bodies alive: 1503
Step 2000/200000 | Bodies alive: 1481 | Merges this step: 2
Step 2100/200000 | Bo

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Circle
from matplotlib.collections import PathCollection
import os
import glob
import csv
from IPython.display import HTML, Video
import warnings
warnings.filterwarnings('ignore')

# ========================== CONFIGURATION ====================================
DATA_DIR = "data_serial"
OUTPUT_VIDEO = "planet_formation_serial.mp4"
FPS = 30
DPI = 150
FIGSIZE = (10, 10)

# Physical scale for plot limits (in AU for display, data is in meters)
AU = 1.496e11  # 1 AU in meters
PLOT_LIMIT_AU = 4.0  # show ±4 AU

# ========================== LOAD SNAPSHOTS ====================================
def load_snapshot(filepath):
    """Load a single CSV snapshot, return arrays of x, y, mass, radius."""
    x, y, mass, radius = [], [], [], []
    with open(filepath, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            x.append(float(row['x']))
            y.append(float(row['y']))
            mass.append(float(row['mass']))
            radius.append(float(row['radius']))
    return np.array(x), np.array(y), np.array(mass), np.array(radius)

def get_sorted_snapshots(data_dir):
    """Get list of snapshot files sorted by step number."""
    files = glob.glob(os.path.join(data_dir, "step_*.csv"))
    files.sort(key=lambda f: int(os.path.basename(f).split('_')[1].split('.')[0]))
    return files

# Load all snapshots
snapshot_files = get_sorted_snapshots(DATA_DIR)
print(f"Found {len(snapshot_files)} snapshots in {DATA_DIR}/")

# Quick peek at first and last snapshot
x0, y0, m0, r0 = load_snapshot(snapshot_files[0])
xf, yf, mf, rf = load_snapshot(snapshot_files[-1])
print(f"Initial: {len(x0)} bodies, mass range [{m0.min():.2e}, {m0.max():.2e}] kg")
print(f"Final:   {len(xf)} bodies, mass range [{mf.min():.2e}, {mf.max():.2e}] kg")

Found 20001 snapshots in data_serial/
Initial: 2001 bodies, mass range [4.98e+24, 1.99e+30] kg
Final:   676 bodies, mass range [5.01e+24, 1.99e+30] kg


In [ ]:
import matplotlib.animation as animation
import sys

# ========================== COLOR MAP =========================================
# Mass-based coloring: small=blue/gray → medium=orange → large=white/yellow
def mass_to_color(masses, global_min_mass, global_max_mass):
    """Map mass values to RGBA colors for astrophysics-style rendering."""
    if global_max_mass <= global_min_mass:
        norm = np.zeros_like(masses)
    else:
        # Log-scale normalization for mass
        log_m = np.log10(np.clip(masses, global_min_mass, global_max_mass))
        log_min = np.log10(global_min_mass)
        log_max = np.log10(global_max_mass)
        if log_max > log_min:
            norm = (log_m - log_min) / (log_max - log_min)
        else:
            norm = np.zeros_like(log_m)

    # Custom colormap: dark blue → orange → bright yellow/white
    colors = np.zeros((len(masses), 4))
    for i, t in enumerate(norm):
        if t < 0.3:
            # Small bodies: steel blue / gray
            s = t / 0.3
            colors[i] = [0.4 + 0.2*s, 0.5 + 0.1*s, 0.7 - 0.1*s, 0.6 + 0.2*s]
        elif t < 0.7:
            # Medium bodies: orange
            s = (t - 0.3) / 0.4
            colors[i] = [0.8 + 0.2*s, 0.5 + 0.2*s, 0.2 - 0.1*s, 0.85 + 0.1*s]
        else:
            # Large bodies: bright yellow → white (planet embryos)
            s = (t - 0.7) / 0.3
            colors[i] = [1.0, 0.8 + 0.2*s, 0.3 + 0.7*s, 1.0]
    return colors

def mass_to_size(masses, global_min_mass, global_max_mass):
    """Map mass to marker size (proportional to radius ~ m^(1/3))."""
    if global_max_mass <= global_min_mass:
        return np.full_like(masses, 2.0)
    log_m = np.log10(np.clip(masses, global_min_mass, global_max_mass))
    log_min = np.log10(global_min_mass)
    log_max = np.log10(global_max_mass)
    if log_max > log_min:
        norm = (log_m - log_min) / (log_max - log_min)
    else:
        norm = np.zeros_like(log_m)
    # Size range: 1 to 120 points
    sizes = 1.0 + 150.0 * (norm ** 1.5)
    return sizes

# ========================== COMPUTE GLOBAL RANGES =============================
# We need global min/max mass across ALL snapshots for consistent coloring
print("Computing global mass range across all snapshots...")
all_masses = []
for sf in snapshot_files[::max(1, len(snapshot_files)//20)]:  # sample every ~20th
    _, _, m, _ = load_snapshot(sf)
    all_masses.append(m)
all_masses = np.concatenate(all_masses)
# Exclude the star (largest mass) for asteroid coloring
star_mass = all_masses.max()
asteroid_masses = all_masses[all_masses < star_mass * 0.5]
GLOBAL_MIN_MASS = asteroid_masses.min() if len(asteroid_masses) > 0 else all_masses.min()
GLOBAL_MAX_MASS = asteroid_masses.max() if len(asteroid_masses) > 0 else all_masses.max()
print(f"Asteroid mass range: [{GLOBAL_MIN_MASS:.2e}, {GLOBAL_MAX_MASS:.2e}] kg")
print(f"Star mass: {star_mass:.2e} kg")

# ========================== RENDER ANIMATION ==================================
total_frames = len(snapshot_files)
print(f"\nRendering animation ({total_frames} frames)...")

fig, ax = plt.subplots(1, 1, figsize=FIGSIZE, facecolor='black')
ax.set_facecolor('black')

# Store trail history
TRAIL_LENGTH = 5
trail_history = []

import time
_render_start_time = time.time()

def render_frame(frame_idx):
    ax.clear()
    ax.set_facecolor('black')

    # ---- Progress reporting ----
    frame_num = frame_idx + 1
    elapsed = time.time() - _render_start_time
    if frame_num > 1:
        fps_actual = (frame_num - 1) / elapsed
        eta = (total_frames - frame_num) / fps_actual
        eta_min, eta_sec = divmod(int(eta), 60)
        print(f"\rRendering frame {frame_num}/{total_frames} "
              f"({100*frame_num/total_frames:.1f}%) | "
              f"{fps_actual:.1f} frames/s | "
              f"ETA: {eta_min}m {eta_sec}s   ", end="")
    else:
        print(f"\rRendering frame {frame_num}/{total_frames} "
              f"({100*frame_num/total_frames:.1f}%)   ", end="")
    sys.stdout.flush()

    # Load snapshot
    x, y, mass, radius = load_snapshot(snapshot_files[frame_idx])

    # Convert to AU for display
    x_au = x / AU
    y_au = y / AU

    # Identify star (most massive body) vs asteroids
    star_idx = np.argmax(mass)

    # Separate star and asteroids
    is_star = np.zeros(len(mass), dtype=bool)
    is_star[star_idx] = True
    ast_mask = ~is_star

    # ---- Draw fading orbital trails ----
    trail_history.append((x_au[ast_mask].copy(), y_au[ast_mask].copy()))
    if len(trail_history) > TRAIL_LENGTH:
        trail_history.pop(0)

    for t_idx, (tx, ty) in enumerate(trail_history[:-1]):
        alpha = 0.05 + 0.1 * (t_idx / max(1, TRAIL_LENGTH))
        n_pts = min(len(tx), 300)  # limit trail points for performance
        ax.scatter(tx[:n_pts], ty[:n_pts], s=0.3, c='steelblue',
                   alpha=alpha, edgecolors='none', rasterized=True)

    # ---- Draw asteroids ----
    ast_x = x_au[ast_mask]
    ast_y = y_au[ast_mask]
    ast_mass = mass[ast_mask]

    colors = mass_to_color(ast_mass, GLOBAL_MIN_MASS, GLOBAL_MAX_MASS)
    sizes = mass_to_size(ast_mass, GLOBAL_MIN_MASS, GLOBAL_MAX_MASS)

    ax.scatter(ast_x, ast_y, s=sizes, c=colors, edgecolors='none',
               alpha=0.9, rasterized=True)

    # ---- Draw star with glow ----
    star_x = x_au[star_idx]
    star_y = y_au[star_idx]

    # Outer glow layers
    for glow_size, glow_alpha in [(800, 0.03), (500, 0.06), (300, 0.1), (150, 0.2)]:
        ax.scatter([star_x], [star_y], s=glow_size, c='gold',
                   alpha=glow_alpha, edgecolors='none', zorder=10)
    # Star core
    ax.scatter([star_x], [star_y], s=80, c='white', alpha=1.0,
               edgecolors='none', zorder=11)

    # ---- Axes and labels ----
    ax.set_xlim(-PLOT_LIMIT_AU, PLOT_LIMIT_AU)
    ax.set_ylim(-PLOT_LIMIT_AU, PLOT_LIMIT_AU)
    ax.set_aspect('equal')
    ax.set_xlabel('x [AU]', color='white', fontsize=12)
    ax.set_ylabel('y [AU]', color='white', fontsize=12)
    ax.tick_params(colors='gray', labelsize=9)
    for spine in ax.spines.values():
        spine.set_color('gray')
        spine.set_linewidth(0.5)

    # ---- Info overlay ----
    step_num = int(os.path.basename(snapshot_files[frame_idx]).split('_')[1].split('.')[0])
    n_bodies = int(ast_mask.sum())
    max_ast_mass = ast_mass.max() if len(ast_mass) > 0 else 0

    info_text = (f"Step: {step_num}\n"
                 f"Bodies: {n_bodies}\n"
                 f"Max asteroid mass: {max_ast_mass:.2e} kg")
    ax.text(0.02, 0.98, info_text, transform=ax.transAxes,
            color='white', fontsize=10, verticalalignment='top',
            fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.7))

    ax.set_title('Protoplanetary Disk — N-Body Simulation (Serial)',
                 color='white', fontsize=14, pad=10)

    return ax,

# Create animation
anim = animation.FuncAnimation(fig, render_frame,
                                frames=total_frames,
                                interval=1000//FPS, blit=False)

# Save as MP4
Writer = animation.FFMpegWriter(fps=FPS, bitrate=3000,
                                 extra_args=['-vcodec', 'libx264', '-pix_fmt', 'yuv420p'])
anim.save(OUTPUT_VIDEO, writer=Writer, dpi=DPI)
plt.close(fig)

total_elapsed = time.time() - _render_start_time
elapsed_min, elapsed_sec = divmod(int(total_elapsed), 60)
print(f"\n\nAnimation saved: {OUTPUT_VIDEO}")
print(f"Frames: {total_frames}, FPS: {FPS}, Duration: {total_frames/FPS:.1f}s")
print(f"Render time: {elapsed_min}m {elapsed_sec}s ({total_frames/total_elapsed:.1f} frames/s)")

Computing global mass range across all snapshots...
Asteroid mass range: [4.98e+24, 6.37e+26] kg
Star mass: 1.99e+30 kg

Rendering animation (20001 frames)...
Rendering frame 20001/20001 (100.0%) | 4.2 frames/s | ETA: 0m 0s   

Animation saved: planet_formation_serial.mp4
Frames: 20001, FPS: 30, Duration: 666.7s
Render time: 78m 35s (4.2 frames/s)


In [ ]:
# Display the video in notebook (works on Colab)
from IPython.display import Video
Video(OUTPUT_VIDEO, embed=True, width=700)

Buffered data was truncated after reaching the output size limit.

In [1]:
!git clone https://github.com/JeganT143/Parallel-Asteroid-Belt-Dynamics-and-Planet-Formation-Simulation.git

fatal: destination path 'Parallel-Asteroid-Belt-Dynamics-and-Planet-Formation-Simulation' already exists and is not an empty directory.


In [2]:
%%writefile nbody_openmp.cpp
// =============================================================================
// Parallel Asteroid Belt Dynamics & Planet Formation Simulation
// OpenMP Parallel Implementation — Velocity-Verlet N-body with Collisions
// =============================================================================

#include <iostream>
#include <fstream>
#include <vector>
#include <cmath>
#include <cstdlib>
#include <string>
#include <algorithm>
#include <chrono>
#include <sstream>
#include <iomanip>
#include <omp.h>

// ========================== CONSTANTS ========================================
const double G             = 6.674e-11;
const double PI            = 3.14159265358979323846;
const double DENSITY       = 3000.0;

// Collision cross-section inflation (applied to asteroids only)
const double RADIUS_INFLATION = 5.0;

// ========================== DATA STRUCTURES ==================================
struct Body {
    double x, y;
    double vx, vy;
    double ax, ay;
    double mass;
    double radius;
    bool   alive;
};

struct SimParams {
    int    num_bodies;
    double star_mass;
    double disk_inner;
    double disk_outer;
    double dt;
    int    num_steps;
    int    snapshot_interval;
    double softening;
    double velocity_perturbation;
    std::string output_dir;
    int    num_threads;
};

// ========================== UTILITY ==========================================
double compute_radius(double mass) {
    return std::pow(3.0 * mass / (4.0 * PI * DENSITY), 1.0 / 3.0);
}

// ========================== INITIAL CONDITIONS ===============================
std::vector<Body> generate_initial_conditions(const SimParams& params) {
    std::vector<Body> bodies;
    bodies.reserve(params.num_bodies + 1);

    Body star;
    star.x = 0.0; star.y = 0.0;
    star.vx = 0.0; star.vy = 0.0;
    star.ax = 0.0; star.ay = 0.0;
    star.mass   = params.star_mass;
    star.radius = compute_radius(params.star_mass);
    star.alive  = true;
    bodies.push_back(star);

    std::srand(42);
    double total_disk_mass = 0.01 * params.star_mass;
    double body_mass = total_disk_mass / params.num_bodies;

    for (int i = 0; i < params.num_bodies; ++i) {
        Body b;
        double u = (double)std::rand() / RAND_MAX;
        double r = params.disk_inner + u * (params.disk_outer - params.disk_inner);
        double theta = 2.0 * PI * ((double)std::rand() / RAND_MAX);

        b.x = r * std::cos(theta);
        b.y = r * std::sin(theta);

        double v_kep = std::sqrt(G * params.star_mass / r);
        double vx_tan = -v_kep * std::sin(theta);
        double vy_tan =  v_kep * std::cos(theta);

        double perturb_vx = params.velocity_perturbation * v_kep *
                            (2.0 * ((double)std::rand() / RAND_MAX) - 1.0);
        double perturb_vy = params.velocity_perturbation * v_kep *
                            (2.0 * ((double)std::rand() / RAND_MAX) - 1.0);

        b.vx = vx_tan + perturb_vx;
        b.vy = vy_tan + perturb_vy;
        b.ax = 0.0; b.ay = 0.0;

        // Small mass variation (0.5x to 2x)
        double mass_factor = 0.5 + 1.5 * ((double)std::rand() / RAND_MAX);
        b.mass   = body_mass * mass_factor;
        b.radius = compute_radius(b.mass) * RADIUS_INFLATION;
        b.alive  = true;

        bodies.push_back(b);
    }
    return bodies;
}

// ========================== PHYSICS: OPENMP FORCE COMPUTATION ================
// Parallelized gravitational force computation
// Each thread computes the FULL acceleration for its assigned bodies
// by iterating over ALL other bodies. This avoids data races entirely.
void compute_forces_omp(std::vector<Body>& bodies, double softening_sq) {
    int n = bodies.size();

    // Reset accelerations (parallel)
    #pragma omp parallel for schedule(static)
    for (int i = 0; i < n; ++i) {
        bodies[i].ax = 0.0;
        bodies[i].ay = 0.0;
    }

    // Parallel force computation — each thread handles its own i values
    // and iterates over ALL j for each i. Thread-safe: only writes to bodies[i].
    #pragma omp parallel for schedule(dynamic, 16)
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;

        double ax_local = 0.0;
        double ay_local = 0.0;

        for (int j = 0; j < n; ++j) {
            if (i == j || !bodies[j].alive) continue;

            double dx = bodies[j].x - bodies[i].x;
            double dy = bodies[j].y - bodies[i].y;
            double dist_sq = dx * dx + dy * dy + softening_sq;
            double inv_dist = 1.0 / std::sqrt(dist_sq);
            double inv_dist3 = inv_dist * inv_dist * inv_dist;

            ax_local += G * bodies[j].mass * dx * inv_dist3;
            ay_local += G * bodies[j].mass * dy * inv_dist3;
        }

        bodies[i].ax = ax_local;
        bodies[i].ay = ay_local;
    }
}

// ========================== INTEGRATOR: VELOCITY-VERLET (PARALLEL) ===========
void integrate_step_omp(std::vector<Body>& bodies, double dt, double softening_sq) {
    int n = bodies.size();

    // Half-kick
    #pragma omp parallel for schedule(static)
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        bodies[i].vx += 0.5 * dt * bodies[i].ax;
        bodies[i].vy += 0.5 * dt * bodies[i].ay;
    }

    // Drift
    #pragma omp parallel for schedule(static)
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        bodies[i].x += dt * bodies[i].vx;
        bodies[i].y += dt * bodies[i].vy;
    }

    // New forces
    compute_forces_omp(bodies, softening_sq);

    // Second half-kick
    #pragma omp parallel for schedule(static)
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        bodies[i].vx += 0.5 * dt * bodies[i].ax;
        bodies[i].vy += 0.5 * dt * bodies[i].ay;
    }
}

// ========================== COLLISION DETECTION & MERGING ====================
// Collisions remain serial — they are O(N^2) but run fast since we check
// only alive bodies and the merge logic is order-dependent.
int handle_collisions(std::vector<Body>& bodies) {
    int n = bodies.size();
    int merges = 0;

    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        if (i == 0) continue;  // star does not participate in merging
        for (int j = i + 1; j < n; ++j) {
            if (!bodies[j].alive) continue;

            double dx = bodies[j].x - bodies[i].x;
            double dy = bodies[j].y - bodies[i].y;
            double dist = std::sqrt(dx * dx + dy * dy);
            double overlap_dist = bodies[i].radius + bodies[j].radius;

            if (dist < overlap_dist) {
                double m1 = bodies[i].mass;
                double m2 = bodies[j].mass;
                double m_total = m1 + m2;

                bodies[i].vx = (m1 * bodies[i].vx + m2 * bodies[j].vx) / m_total;
                bodies[i].vy = (m1 * bodies[i].vy + m2 * bodies[j].vy) / m_total;
                bodies[i].x = (m1 * bodies[i].x + m2 * bodies[j].x) / m_total;
                bodies[i].y = (m1 * bodies[i].y + m2 * bodies[j].y) / m_total;

                bodies[i].mass   = m_total;
                bodies[i].radius = compute_radius(m_total) * RADIUS_INFLATION;
                bodies[j].alive = false;
                ++merges;
            }
        }
    }
    return merges;
}

// ========================== I/O ==============================================
void write_snapshot(const std::vector<Body>& bodies, int step, const std::string& dir) {
    std::ostringstream filename;
    filename << dir << "/step_" << std::setw(6) << std::setfill('0') << step << ".csv";

    std::ofstream file(filename.str());
    if (!file.is_open()) {
        std::cerr << "ERROR: Cannot open " << filename.str() << std::endl;
        return;
    }

    file << "id,x,y,vx,vy,mass,radius\n";
    int id = 0;
    for (size_t i = 0; i < bodies.size(); ++i) {
        if (!bodies[i].alive) continue;
        file << id << ","
             << std::scientific << std::setprecision(8)
             << bodies[i].x << ","
             << bodies[i].y << ","
             << bodies[i].vx << ","
             << bodies[i].vy << ","
             << bodies[i].mass << ","
             << bodies[i].radius << "\n";
        ++id;
    }
    file.close();
}

int count_alive(const std::vector<Body>& bodies) {
    int count = 0;
    for (size_t i = 0; i < bodies.size(); ++i) {
        if (bodies[i].alive) ++count;
    }
    return count;
}

// ========================== MAIN =============================================
int main(int argc, char* argv[]) {
    SimParams params;
    params.num_bodies             = 2000;
    params.star_mass              = 1.989e30;
    params.disk_inner             = 1.0e11;
    params.disk_outer             = 5.0e11;
    params.dt                     = 1.0e6;
    params.num_steps              = 2000;
    params.snapshot_interval      = 10;
    params.softening              = 1.0e9;
    params.velocity_perturbation  = 0.10;
    params.output_dir             = "data_openmp";
    params.num_threads            = 4;

    for (int i = 1; i < argc; ++i) {
        std::string arg = argv[i];
        if (arg == "--bodies"    && i+1 < argc) params.num_bodies        = std::atoi(argv[++i]);
        else if (arg == "--steps"     && i+1 < argc) params.num_steps         = std::atoi(argv[++i]);
        else if (arg == "--dt"        && i+1 < argc) params.dt               = std::atof(argv[++i]);
        else if (arg == "--interval"  && i+1 < argc) params.snapshot_interval = std::atoi(argv[++i]);
        else if (arg == "--outdir"    && i+1 < argc) params.output_dir       = argv[++i];
        else if (arg == "--threads"   && i+1 < argc) params.num_threads      = std::atoi(argv[++i]);
    }

    // Set OpenMP thread count
    omp_set_num_threads(params.num_threads);

    double softening_sq = params.softening * params.softening;

    std::cout << "=== N-Body Asteroid Belt Simulation (OpenMP) ===" << std::endl;
    std::cout << "Bodies:  " << params.num_bodies << std::endl;
    std::cout << "Steps:   " << params.num_steps  << std::endl;
    std::cout << "Threads: " << params.num_threads << std::endl;
    std::cout << "dt:      " << params.dt << " s" << std::endl;
    std::cout << "Output:  " << params.output_dir << "/" << std::endl;

    std::string cmd = "mkdir -p " + params.output_dir;
    system(cmd.c_str());

    auto bodies = generate_initial_conditions(params);
    std::cout << "Generated " << bodies.size() << " bodies (1 star + "
              << params.num_bodies << " asteroids)" << std::endl;

    // Verify thread count
    #pragma omp parallel
    {
        #pragma omp single
        std::cout << "OpenMP active threads: " << omp_get_num_threads() << std::endl;
    }

    compute_forces_omp(bodies, softening_sq);
    write_snapshot(bodies, 0, params.output_dir);

    // ----- Main simulation loop -----
    double t_start = omp_get_wtime();

    for (int step = 1; step <= params.num_steps; ++step) {
        integrate_step_omp(bodies, params.dt, softening_sq);
        int merges = handle_collisions(bodies);

        if (step % params.snapshot_interval == 0) {
            write_snapshot(bodies, step, params.output_dir);
        }

        if (step % 100 == 0 || step == params.num_steps) {
            int alive = count_alive(bodies);
            std::cout << "Step " << step << "/" << params.num_steps
                      << " | Bodies alive: " << alive;
            if (merges > 0) std::cout << " | Merges: " << merges;
            std::cout << std::endl;
        }
    }

    double t_end = omp_get_wtime();
    double elapsed = t_end - t_start;

    std::cout << "\n=== Simulation Complete ===" << std::endl;
    std::cout << "Total time: " << std::fixed << std::setprecision(3) << elapsed << " s" << std::endl;
    std::cout << "Final alive bodies: " << count_alive(bodies) << std::endl;
    std::cout << "Threads used: " << params.num_threads << std::endl;

    return 0;
}

Writing nbody_openmp.cpp


In [3]:
%%writefile nbody_benchmark.cpp
// =============================================================================
// Benchmark harness — measures simulation loop time for Serial vs OpenMP
// =============================================================================
#include <iostream>
#include <vector>
#include <cmath>
#include <cstdlib>
#include <string>
#include <chrono>
#include <iomanip>
#include <omp.h>

const double G  = 6.674e-11;
const double PI = 3.14159265358979323846;
const double DENSITY = 3000.0;
const double RADIUS_INFLATION = 100.0;

struct Body {
    double x, y, vx, vy, ax, ay, mass, radius;
    bool alive;
};

double compute_radius(double mass) {
    return std::pow(3.0 * mass / (4.0 * PI * DENSITY), 1.0 / 3.0);
}

std::vector<Body> generate_ic(int num_bodies, double star_mass) {
    std::vector<Body> bodies;
    bodies.reserve(num_bodies + 1);

    Body star = {0,0,0,0,0,0, star_mass, compute_radius(star_mass), true};
    bodies.push_back(star);

    std::srand(42);
    double body_mass = 0.01 * star_mass / num_bodies;
    double inner = 1.0e11, outer = 5.0e11;

    for (int i = 0; i < num_bodies; ++i) {
        Body b;
        double u = (double)std::rand() / RAND_MAX;
        double r = inner + u * (outer - inner);
        double theta = 2.0 * PI * ((double)std::rand() / RAND_MAX);
        b.x = r * std::cos(theta);
        b.y = r * std::sin(theta);
        double v_kep = std::sqrt(G * star_mass / r);
        b.vx = -v_kep * std::sin(theta) + 0.10*v_kep*(2.0*((double)std::rand()/RAND_MAX)-1.0);
        b.vy =  v_kep * std::cos(theta) + 0.10*v_kep*(2.0*((double)std::rand()/RAND_MAX)-1.0);
        b.ax = 0; b.ay = 0;
        double mf = 0.5 + 1.5 * ((double)std::rand() / RAND_MAX);
        b.mass = body_mass * mf;
        b.radius = compute_radius(b.mass) * RADIUS_INFLATION;
        b.alive = true;
        bodies.push_back(b);
    }
    return bodies;
}

// Serial force computation (Newton's 3rd law)
void compute_forces_serial(std::vector<Body>& bodies, double eps2) {
    int n = bodies.size();
    for (int i = 0; i < n; ++i) { bodies[i].ax = 0; bodies[i].ay = 0; }
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        for (int j = i+1; j < n; ++j) {
            if (!bodies[j].alive) continue;
            double dx = bodies[j].x - bodies[i].x;
            double dy = bodies[j].y - bodies[i].y;
            double d2 = dx*dx + dy*dy + eps2;
            double inv = 1.0/std::sqrt(d2);
            double inv3 = inv*inv*inv;
            double fx = G * dx * inv3;
            double fy = G * dy * inv3;
            bodies[i].ax += fx * bodies[j].mass;
            bodies[i].ay += fy * bodies[j].mass;
            bodies[j].ax -= fx * bodies[i].mass;
            bodies[j].ay -= fy * bodies[i].mass;
        }
    }
}

// OpenMP force computation
void compute_forces_omp(std::vector<Body>& bodies, double eps2) {
    int n = bodies.size();
    #pragma omp parallel for schedule(static)
    for (int i = 0; i < n; ++i) { bodies[i].ax = 0; bodies[i].ay = 0; }

    #pragma omp parallel for schedule(dynamic, 16)
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        double ax_l = 0, ay_l = 0;
        for (int j = 0; j < n; ++j) {
            if (i == j || !bodies[j].alive) continue;
            double dx = bodies[j].x - bodies[i].x;
            double dy = bodies[j].y - bodies[i].y;
            double d2 = dx*dx + dy*dy + eps2;
            double inv = 1.0/std::sqrt(d2);
            double inv3 = inv*inv*inv;
            ax_l += G * bodies[j].mass * dx * inv3;
            ay_l += G * bodies[j].mass * dy * inv3;
        }
        bodies[i].ax = ax_l;
        bodies[i].ay = ay_l;
    }
}

typedef void (*ForceFunc)(std::vector<Body>&, double);

double run_benchmark(int num_bodies, int steps, ForceFunc force_fn, double eps2) {
    auto bodies = generate_ic(num_bodies, 1.989e30);
    double dt = 1.0e6;

    force_fn(bodies, eps2);

    double t0 = omp_get_wtime();
    for (int s = 0; s < steps; ++s) {
        // Half-kick
        for (auto& b : bodies) { if (!b.alive) continue; b.vx += 0.5*dt*b.ax; b.vy += 0.5*dt*b.ay; }
        // Drift
        for (auto& b : bodies) { if (!b.alive) continue; b.x += dt*b.vx; b.y += dt*b.vy; }
        // Forces
        force_fn(bodies, eps2);
        // Half-kick
        for (auto& b : bodies) { if (!b.alive) continue; b.vx += 0.5*dt*b.ax; b.vy += 0.5*dt*b.ay; }
    }
    double t1 = omp_get_wtime();
    return t1 - t0;
}

int main() {
    double eps2 = 1.0e18; // softening^2
    int bench_steps = 200;
    int body_counts[] = {500, 1000, 2000};
    int thread_counts[] = {1, 2, 4};

    std::cout << "=== N-Body Benchmark: Serial vs OpenMP ===" << std::endl;
    std::cout << std::setw(8) << "N"
              << std::setw(15) << "Serial(s)";
    for (int t : thread_counts)
        std::cout << std::setw(12) << "OMP-" + std::to_string(t) + "T(s)";
    for (int t : thread_counts)
        std::cout << std::setw(12) << "Speedup-" + std::to_string(t) + "T";
    std::cout << std::endl;

    // CSV output for Python plotting
    std::cout << "\n#CSV_START" << std::endl;
    std::cout << "N,serial";
    for (int t : thread_counts) std::cout << ",omp_" << t << "t";
    std::cout << std::endl;

    for (int N : body_counts) {
        // Serial
        omp_set_num_threads(1);
        double t_serial = run_benchmark(N, bench_steps, compute_forces_serial, eps2);

        std::cout << N << "," << std::fixed << std::setprecision(3) << t_serial;

        double omp_times[3];
        for (int ti = 0; ti < 3; ++ti) {
            omp_set_num_threads(thread_counts[ti]);
            omp_times[ti] = run_benchmark(N, bench_steps, compute_forces_omp, eps2);
            std::cout << "," << std::fixed << std::setprecision(3) << omp_times[ti];
        }
        std::cout << std::endl;

        // Print table row
        std::cerr << std::setw(8) << N
                  << std::setw(15) << std::fixed << std::setprecision(3) << t_serial;
        for (int ti = 0; ti < 3; ++ti)
            std::cerr << std::setw(12) << std::fixed << std::setprecision(3) << omp_times[ti];
        for (int ti = 0; ti < 3; ++ti)
            std::cerr << std::setw(12) << std::fixed << std::setprecision(2) << t_serial / omp_times[ti];
        std::cerr << std::endl;
    }
    std::cout << "#CSV_END" << std::endl;

    return 0;
}

Writing nbody_benchmark.cpp


In [4]:
!g++ -O2 -fopenmp nbody_openmp.cpp -o nbody_openmp

nbody_openmp.cpp: In function ‘int main(int, char**)’:
nbody_openmp.cpp:291:11: warning: ignoring return value of ‘int system(const char*)’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
  291 |     system(cmd.c_str());
      |     ~~~~~~^~~~~~~~~~~~~


In [5]:
!./nbody_openmp

=== N-Body Asteroid Belt Simulation (OpenMP) ===
Bodies:  2000
Steps:   2000
Threads: 4
dt:      1e+06 s
Output:  data_openmp/
Generated 2001 bodies (1 star + 2000 asteroids)
OpenMP active threads: 4
Step 100/2000 | Bodies alive: 1991
Step 200/2000 | Bodies alive: 1987
Step 300/2000 | Bodies alive: 1977
Step 400/2000 | Bodies alive: 1970
Step 500/2000 | Bodies alive: 1962
Step 600/2000 | Bodies alive: 1956
Step 700/2000 | Bodies alive: 1948
Step 800/2000 | Bodies alive: 1940
Step 900/2000 | Bodies alive: 1932
Step 1000/2000 | Bodies alive: 1921
Step 1100/2000 | Bodies alive: 1913
Step 1200/2000 | Bodies alive: 1905
Step 1300/2000 | Bodies alive: 1900
Step 1400/2000 | Bodies alive: 1888
Step 1500/2000 | Bodies alive: 1880
Step 1600/2000 | Bodies alive: 1875
Step 1700/2000 | Bodies alive: 1868
Step 1800/2000 | Bodies alive: 1862
Step 1900/2000 | Bodies alive: 1852
Step 2000/2000 | Bodies alive: 1845

=== Simulation Complete ===
Total time: 65.852 s
Final alive bodies: 1845
Threads used: 

In [6]:
!g++ -O2 -fopenmp nbody_benchmark.cpp -o benchmark

In [7]:
!./benchmark

=== N-Body Benchmark: Serial vs OpenMP ===
       N      Serial(s)   OMP-1T(s)   OMP-2T(s)   OMP-4T(s)  Speedup-1T  Speedup-2T  Speedup-4T

#CSV_START
N,serial,omp_1t,omp_2t,omp_4t
500,0.243,0.285,0.260,0.272
     500          0.243       0.285       0.260       0.272        0.85        0.94        0.89
1000,0.939,1.135,1.024,1.065
    1000          0.939       1.135       1.024       1.065        0.83        0.92        0.88
2000,4.609,4.491,4.374,5.183
    2000          4.609       4.491       4.374       5.183        1.03        1.05        0.89
#CSV_END


In [8]:
%cd Parallel-Asteroid-Belt-Dynamics-and-Planet-Formation-Simulation

/content/Parallel-Asteroid-Belt-Dynamics-and-Planet-Formation-Simulation


In [9]:
!ls

openMP.ipynb  seres.ipynb


In [10]:
!git add openMP.ipynb

In [11]:
!git add seres.ipynb

In [12]:
!git commit -m "Added OpenMP notebooks"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@6181ef17b0e7.(none)')


In [13]:
!git config --global user.name "UmodaKahatagahawatta"

In [14]:
!git config --global user.email "piyumiumoda@gmail.com"

In [15]:
!git commit -m "Added OpenMP notebooks"

On branch kalhara/seres
Your branch is up to date with 'origin/kalhara/seres'.

nothing to commit, working tree clean


In [16]:
%cd /content/Parallel-Asteroid-Belt-Dynamics-and-Planet-Formation-Simulation

/content/Parallel-Asteroid-Belt-Dynamics-and-Planet-Formation-Simulation


In [17]:
!git branch

* kalhara/seres


In [18]:
!git checkout -b umoda

Switched to a new branch 'umoda'


In [19]:
!git add openMP.ipynb seres.ipynb

In [20]:
!git commit -m "Add OpenMP notebooks"

On branch umoda
nothing to commit, working tree clean


In [21]:
!git push origin umoda

fatal: could not read Username for 'https://github.com': No such device or address
